# ⚡ Module 06: The Modern Transformer Revolution & Attention Mechanics (2017 - 2024)
### *From "Attention Is All You Need" to RoPE, FlashAttention, and the KV-Cache*

---

## 📜 Historical Revolution: The Death of Recurrence
For over twenty years, the consensus was that temporal sequences *required* recurrent connections ( = f(h_{t-1}, x_t)$).
Recurrence came with an insurmountable hardware bottleneck: **The (N)$ Sequential Dependency Barrier**.
Because step $ strictly depends on step -1$, modern GPUs with tens of thousands of CUDA cores were forced to idle, executing steps in serial.

In 2017, **Vaswani et al.** published *"Attention Is All You Need"*, introducing **The Transformer**.
By eliminating recurrence and convolutions entirely, self-attention allowed every token in a sequence of length $ to communicate with every other token in parallel with (1)$ sequential operations!

### 💼 The Sensory Analogy: The Summit of Diplomats
- Every token in a sentence is a **diplomat** holding three envelopes:
  1. **Query ($)**: "What information do I seek?"
  2. **Key ($)**: "What topic/expertise do I possess?"
  3. **Value ($)**: "What detailed knowledge do I contain?"
- Each diplomat compares their Query against everyone else's Key via dot-product resonance ( K^	op$).
- The scores are scaled by $rac{1}{\sqrt{d_k}}$ to prevent softmax saturation, normalized via Softmax, and used to mix the Value envelopes.

## 1. Pure NumPy Scaled Dot-Product & Multi-Head Attention
Let us compute self-attention from scratch and visualize how distinct attention heads learn specialized linguistic roles.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../python/module06_transformers_and_attention"))
import numpy as np
import matplotlib.pyplot as plt
from attention_scratch import scaled_dot_product_attention_scratch, MultiHeadAttentionScratch

tokens = ["The", "animal", "didn't", "cross", "the", "street", "because", "it", "was", "too", "tired"]
seq_len = len(tokens)
d_model = 32
num_heads = 4

mha = MultiHeadAttentionScratch(d_model=d_model, num_heads=num_heads, seed=42)
X = np.random.randn(1, seq_len, d_model)

out, weights = mha.forward(X, is_causal=False)

# Visualize attention patterns across 2 different heads
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor="#0b0f19")
for head_idx, ax in enumerate(axes):
    ax.set_facecolor("#0f172a")
    mat = weights[0, head_idx]
    im = ax.imshow(mat, cmap="viridis", vmin=0.0, vmax=np.max(mat))
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(tokens, rotation=45, ha="right", color="white", fontsize=10)
    ax.set_yticklabels(tokens, color="white", fontsize=10)
    ax.set_title(f"Attention Head {head_idx} Specialization", color="white", fontsize=12, fontweight="bold")
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(colors="#94a3b8")

plt.suptitle("Multi-Head Self-Attention Patterns (Vaswani et al. 2017)", color="white", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 2. Rotary Position Embeddings (RoPE): Rotating in 2D Subspaces

### The Relative Distance Theorem (Su et al. 2021):
RoPE rotates query and key vectors in 2D coordinate pairs by angles proportional to token position $:
17109R_{\Theta, m} = egin{pmatrix} \cos(m	heta) & -\sin(m	heta) \ \sin(m	heta) & \cos(m	heta) \end{pmatrix}17109
Because rotation matrices are orthogonal (^	op R_n = R_{n-m}$), the dot product depends **strictly on relative distance** $:
17109\langle R_m q, R_n k angle = q^	op R_{n-m} k17109

Let us plot how rotating vector $ across positions =0, 1, 2, \dots, 8$ causes the inner product to smoothly decay with distance.

In [ ]:
import torch
from rope_and_kv_cache import rope_relative_distance_inner_product, precompute_rope_freqs

# Compute inner product as a function of relative distance |m - n|
torch.manual_seed(42)
dim = 64
q = torch.randn(dim)
k = torch.randn(dim)

distances = np.arange(0, 50)
inner_prods = [rope_relative_distance_inner_product(q, k, pos_m=d, pos_n=0, dim=dim) for d in distances]

plt.figure(figsize=(10, 4.5), facecolor="#0b0f19")
ax = plt.gca()
ax.set_facecolor("#0f172a")

plt.plot(distances, inner_prods, color="#a855f7", linewidth=2.5, marker="o", markersize=4, label="RoPE <R_m q, R_0 k> Inner Product")
plt.axhline(0, color="#64748b", linestyle="--", alpha=0.6)
plt.title("RoPE Relative Distance Decay Property (Su et al. 2021)", color="white", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Relative Token Distance |m - n|", color="#94a3b8", fontsize=11)
plt.ylabel("Dot Product Resonance", color="#94a3b8", fontsize=11)
plt.grid(True, ls="--", color="#334155", alpha=0.4)
plt.legend(facecolor="#1e293b", edgecolor="#475569", labelcolor="white", fontsize=11)
plt.tick_params(colors="#94a3b8")
plt.tight_layout()
plt.show()

print("Notice how the interference of high and low frequency rotations provides a natural locality prior!")


## 3. Autoregressive KV-Cache: Taming the (T^3)$ Generation Bottleneck

During generation (inference), generating $ tokens:
- **Without KV-Cache**: Step $ computes $ and $ for all past tokens  \dots t$ $\implies \sum_{t=1}^T t^2 = O(T^3)$ operations.
- **With KV-Cache**: Past $ and $ are cached in high-speed buffer. Step $ only computes , k_t, v_t$ for the single new token $\implies \sum_{t=1}^T t = O(T^2)$ operations (a 	imes - 1000	imes$ speedup!).

In [ ]:
from transformer_block_pytorch import TransformerDecoderBlock, KVCache

dim = 64
num_heads = 4
num_kv_heads = 2  # Grouped Query Attention (GQA)
block = TransformerDecoderBlock(dim=dim, num_heads=num_heads, num_kv_heads=num_kv_heads)
block.eval()

seq_len = 8
x = torch.randn(1, seq_len, dim)
pos_ids = torch.arange(seq_len).unsqueeze(0)

# Verify KV-Cache generates exact same representations as full parallel pass
with torch.no_grad():
    out_parallel = block(x, pos_ids, is_causal=True)

cache = KVCache(num_layers=1)
cached_steps = []
with torch.no_grad():
    for t in range(seq_len):
        tok_out = block(x[:, t:t+1, :], pos_ids[:, t:t+1], kv_cache=cache, layer_idx=0)
        cached_steps.append(tok_out)

out_cached = torch.cat(cached_steps, dim=1)
diff = torch.max(torch.abs(out_parallel - out_cached)).item()
print(f"Max difference between parallel forward and KV-Cache decoding: {diff:.2e}")
print("✅ Production SOTA Transformer Decoder Block with RMSNorm, SwiGLU, and GQA verified!")
